In [2]:
#========================
#AIPL 2026 Predection ML
#========================
#ALL librires are imported here

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss

from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings("ignore")

In [4]:
#========================
#LOAD the data
#========================
train = pd.read_csv("train_IPL.csv")
schedule = pd.read_csv("schedule.csv")
sample = pd.read_csv("sample_submission.csv")

print(train.shape)
print(schedule.shape)
print(sample.shape)

train.head()


(272704, 38)
(5, 6)
(53, 5)


,Match ID,Date,Venue,Bat First,Bat Second,Innings,Over,Ball,Batter,Non Striker,...,Player Out Runs,Player Out Balls Faced,Bowler Runs Conceded,Valid Ball,toss_winner,toss_decision,city,result_type,season,match_won_by
0,335982,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,1,SC Ganguly,BB McCullum,...,NaN,NaN,0,1,Royal Challengers Bangalore,field,Bengaluru,NaN,2007/08,Kolkata Knight Riders
1,335982,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,2,BB McCullum,SC Ganguly,...,NaN,NaN,0,1,Royal Challengers Bangalore,field,Bengaluru,NaN,2007/08,Kolkata Knight Riders
2,335982,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,3,BB McCullum,SC Ganguly,...,NaN,NaN,1,0,Royal Challengers Bangalore,field,Bengaluru,NaN,2007/08,Kolkata Knight Riders
3,335982,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,3,BB McCullum,SC Ganguly,...,NaN,NaN,0,1,Royal Challengers Bangalore,field,Bengaluru,NaN,2007/08,Kolkata Knight Riders
4,335982,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,4,BB McCullum,SC Ganguly,...,NaN,NaN,0,1,Royal Challengers Bangalore,field,Bengaluru,NaN,2007/08,Kolkata Knight Riders


In [5]:
#========================
#Create the Match data
#========================
# First innings final score
inn1 = (
    train[train["Innings"] == 1]
    .groupby("Match ID")
    .tail(1)[["Match ID", "Innings Runs"]]
    .rename(columns={"Innings Runs": "inn1_runs"})
)

# Second innings final score
inn2 = (
    train[train["Innings"] == 2]
    .groupby("Match ID")
    .tail(1)[["Match ID", "Innings Runs", "Innings Wickets"]]
    .rename(columns={
        "Innings Runs": "inn2_runs",
        "Innings Wickets": "inn2_wickets"
    })
)

# Match basic info
match_df = (
    train.groupby("Match ID")
    .first()
    .reset_index()
)
# Merge all data
match_df = match_df.merge(inn1, on="Match ID", how="left")
match_df = match_df.merge(inn2, on="Match ID", how="left")

print(match_df.shape)

match_df.head()


(1145, 41)


,Match ID,Date,Venue,Bat First,Bat Second,Innings,Over,Ball,Batter,Non Striker,...,Valid Ball,toss_winner,toss_decision,city,result_type,season,match_won_by,inn1_runs,inn2_runs,inn2_wickets
0,335982,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,1,SC Ganguly,BB McCullum,...,1,Royal Challengers Bangalore,field,Bengaluru,NaN,2007/08,Kolkata Knight Riders,222,82.0,10.0
1,335983,2008-04-19,Punjab Cricket Association Stadium,Chennai Super Kings,Kings XI Punjab,1,1,1,PA Patel,ML Hayden,...,1,Chennai Super Kings,bat,Chandigarh,NaN,2007/08,Chennai Super Kings,240,207.0,4.0
2,335984,2008-04-19,Feroz Shah Kotla,Rajasthan Royals,Delhi Daredevils,1,1,1,T Kohli,YK Pathan,...,1,Rajasthan Royals,bat,Delhi,NaN,2007/08,Delhi Daredevils,129,132.0,1.0
3,335985,2008-04-20,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,1,1,1,L Ronchi,ST Jayasuriya,...,1,Mumbai Indians,bat,Mumbai,NaN,2007/08,Royal Challengers Bangalore,165,166.0,5.0
4,335986,2008-04-20,Eden Gardens,Deccan Chargers,Kolkata Knight Riders,1,1,1,AC Gilchrist,Y Venugopal Rao,...,1,Deccan Chargers,bat,Kolkata,NaN,2007/08,Kolkata Knight Riders,110,112.0,5.0


In [6]:
#========================
# 4. REMOVE TIES / NO RESULTS
#========================

match_df = match_df[
    (~match_df["result_type"].isin(["tie", "no result"]))
]

print(match_df.shape)

(1124, 41)


In [8]:
# ===========================
# 5. CREATE TARGET LABELS
# ===========================

def create_label(row):

    winner = row["match_won_by"]

    team_a = row["Bat First"]
    team_b = row["Bat Second"]

    inn1_runs = row["inn1_runs"]
    inn2_runs = row["inn2_runs"]

    wickets_lost = row["inn2_wickets"]

     # Team A wins
    if winner == team_a:

        margin = inn1_runs - inn2_runs

        if margin <= 20:
            return "A_small"
        else:
            return "A_big"
    # Team B wins
    elif winner == team_b:

        wickets_remaining = 10 - wickets_lost

        if wickets_remaining <= 5:
            return "B_small"
        else:
            return "B_big"

    else:
        return np.nan
    
    
match_df["target"] = match_df.apply(create_label, axis=1)

match_df = match_df.dropna(subset=["target"])

print(match_df["target"].value_counts())

target
B_big      401
A_big      273
A_small    240
B_small    210
Name: count, dtype: int64


In [9]:
#========================
# 6. FEATURE ENGINEERING
#========================

# Match level features
features = [
    "Venue",
    "city",
    "Bat First",
    "Bat Second",
    "toss_winner",
    "toss_decision",
    "season"
]

data = match_df[features + ["target"]].copy()

data.head()

,Venue,city,Bat First,Bat Second,toss_winner,toss_decision,season,target
0,M Chinnaswamy Stadium,Bengaluru,Kolkata Knight Riders,Royal Challengers Bangalore,Royal Challengers Bangalore,field,2007/08,A_big
1,Punjab Cricket Association Stadium,Chandigarh,Chennai Super Kings,Kings XI Punjab,Chennai Super Kings,bat,2007/08,A_big
2,Feroz Shah Kotla,Delhi,Rajasthan Royals,Delhi Daredevils,Rajasthan Royals,bat,2007/08,B_big
3,Wankhede Stadium,Mumbai,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,2007/08,B_small
4,Eden Gardens,Kolkata,Deccan Chargers,Kolkata Knight Riders,Deccan Chargers,bat,2007/08,B_small


In [10]:
#========================
# 7. LABEL ENCODING
#========================

encoders = {}

for col in features:

    le = LabelEncoder()

    data[col] = le.fit_transform(data[col].astype(str))

    encoders[col] = le


# Encode target
target_encoder = LabelEncoder()

data["target"] = target_encoder.fit_transform(data["target"])

print(target_encoder.classes_)

['A_big' 'A_small' 'B_big' 'B_small']


In [11]:
#========================
# 8. TRAIN TEST SPLIT
#========================

X = data[features]
y = data["target"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_valid.shape)



(899, 7)
(225, 7)


In [12]:
#========================
# 9. TRAIN MODEL
#========================
model = LGBMClassifier(
    objective="multiclass",
    num_class=4,
    n_estimators=300,
    learning_rate=0.03,
    max_depth=6,
    random_state=42
)

model.fit(X_train, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000232 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 146
[LightGBM] [Info] Number of data points in the train set: 899, number of used features: 7
[LightGBM] [Info] Start training from score -1.416788
[LightGBM] [Info] Start training from score -1.543788
[LightGBM] [Info] Start training from score -1.029842
[LightGBM] [Info] Start training from score -1.677319
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.03
,n_estimators,300
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [13]:
#========================
# 10. VALIDATION
#========================
preds = model.predict_proba(X_valid)

score = log_loss(y_valid, preds)

print("Validation Log Loss:", score)

Validation Log Loss: 1.6718045876706569


In [17]:
#========================
# 11. PREPARE FUTURE MATCH DATA
#========================

schedule.head()
# Example expected columns:
#

future = schedule.copy()

# Dummy toss placeholders
future["toss_winner"] = future["team_a"]
future["toss_decision"] = "field"

future = future.rename(columns={
    "team_a": "Bat First",
    "team_b": "Bat Second"
})
future["toss_decision"] = "field"

future = future.rename(columns={
    "Team_A": "Bat First",
    "Team_B": "Bat Second"
})

In [20]:
#========================
# 12. APPLY ENCODERS
#========================

future["Venue"] = future["venue"]
future["season"] = future["date"].astype(str).str[:4]

future_data = future[features].copy()

for col in features:

    le = encoders[col]

    # Handle unseen labels
    future_data[col] = future_data[col].astype(str)

    known = set(le.classes_)

    future_data[col] = future_data[col].apply(
        lambda x: x if x in known else le.classes_[0]
    )

    future_data[col] = le.transform(future_data[col])


future_data.head()

,Venue,city,Bat First,Bat Second,toss_winner,toss_decision,season
0,18,7,0,5,0,1,0
1,27,15,18,17,18,1,0
2,3,23,9,12,9,1,0
3,37,25,10,13,10,1,0
4,10,22,8,2,8,1,0


In [21]:
#========================
# 13. PREDICT PROBABILITIES
#========================
future_preds = model.predict_proba(future_data)

future_preds[:5]

array([[0.15232694, 0.50416853, 0.30729752, 0.03620701],
       [0.09326565, 0.05618894, 0.57348134, 0.27706407],
       [0.073028  , 0.27334918, 0.43706055, 0.21656227],
       [0.25911965, 0.06171637, 0.42473072, 0.25443327],
       [0.26955144, 0.13002232, 0.55971116, 0.04071508]])

In [22]:
# ============================================================
# 14. CREATE SUBMISSION
# ============================================================

submission = pd.DataFrame()

submission["match_id"] = schedule["match_id"]

class_names = target_encoder.classes_

for i, cls in enumerate(class_names):
    submission[cls] = future_preds[:, i]

submission.head()


,match_id,A_big,A_small,B_big,B_small
0,M_2026_T01,0.152327,0.504169,0.307298,0.036207
1,M_2026_T02,0.093266,0.056189,0.573481,0.277064
2,M_2026_T03,0.073028,0.273349,0.437061,0.216562
3,M_2026_T04,0.259120,0.061716,0.424731,0.254433
4,M_2026_T05,0.269551,0.130022,0.559711,0.040715


In [23]:
# ============================================================
# 15. ENSURE ALL 4 COLUMNS EXIST
# ============================================================

required_cols = ["A_small", "A_big", "B_small", "B_big"]

for col in required_cols:

    if col not in submission.columns:
        submission[col] = 0.0


submission = submission[
    ["match_id", "A_small", "A_big", "B_small", "B_big"]
]

submission.head()


,match_id,A_small,A_big,B_small,B_big
0,M_2026_T01,0.504169,0.152327,0.036207,0.307298
1,M_2026_T02,0.056189,0.093266,0.277064,0.573481
2,M_2026_T03,0.273349,0.073028,0.216562,0.437061
3,M_2026_T04,0.061716,0.259120,0.254433,0.424731
4,M_2026_T05,0.130022,0.269551,0.040715,0.559711


In [ ]:
# ============================================================
# CREATE FINAL SUBMISSION
# ============================================================

submission = sample.copy()

# Predict probabilities for future matches
future_preds = model.predict_proba(future_data)

# Fill ONLY the last 5 rows
# (Future IPL 2026 fixtures)

submission.loc[
    submission.index[-5:],
    ["A_small", "A_big", "B_small", "B_big"]
] = future_preds

# ------------------------------------------------------------
# For remaining 48 rows
# Use class prior probabilities
# ------------------------------------------------------------

class_probs = (
    match_df["target"]
    .value_counts(normalize=True)
)

submission.loc[
    submission.index[:-5],
    "A_small"
] = class_probs.get("A_small", 0)

submission.loc[
    submission.index[:-5],
    "A_big"
] = class_probs.get("A_big", 0)

submission.loc[
    submission.index[:-5],
    "B_small"
] = class_probs.get("B_small", 0)

submission.loc[
    submission.index[:-5],
    "B_big"
] = class_probs.get("B_big", 0)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    submission.to_csv(f, index=False)

print(submission.shape)
print("submission.csv created successfully")

PermissionError: [Errno 13] Permission denied: 'submission.csv'

In [25]:
# ============================================================
# 17. FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

         Feature  Importance
6         season        5098
2      Bat First        3775
3     Bat Second        3574
0          Venue        3060
4    toss_winner        2821
1           city        2814
5  toss_decision         447
